# Data Understanding: All Nine Olist Datasets

## Purpose

This notebook documents the data-understanding phase for the E-Commerce Delivery Intelligence project. It starts with the orders dataset because orders form the central table, preserves the completed Tasks 1–4, and then extends the same read-only investigation to all nine Olist source datasets.

The notebook covers source loading, schemas, grain, missingness, duplicates, candidate identifiers, relationships, and potential ML relevance. It only inspects the data; it does not clean, transform, create targets, build production joins, or train models.

## 1. Load the Orders Data

`Path.cwd()` gets the directory from which the notebook is running. Because this notebook is intended to run from `notebooks/`, its parent is the project root. Building the path with `pathlib` keeps it portable and avoids a machine-specific absolute path.

The existence check raises a clear error containing the expected location if the CSV is missing. `pd.read_csv(...)` then reads the original file without modifying it.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
orders_path = project_root / "data" / "raw" / "olist_orders_dataset.csv"

if not orders_path.exists():
    raise FileNotFoundError(f"Orders dataset not found at expected path: {orders_path}")

orders = pd.read_csv(orders_path)

### 1.1 Preview the Loaded Orders

`orders.head()` displays the first five rows by default. This small preview confirms that the DataFrame loaded and lets us see the column layout without starting data-quality analysis or other EDA tasks.

In [2]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 2. Structural Inspection

Structural inspection helps us understand the dataset's size, the information available in its columns, and how Pandas currently represents that information. Establishing this structure gives us a clear foundation before any deeper analysis.

### 2.1 Dataset Shape

`DataFrame.shape` returns a tuple in the form `(number_of_rows, number_of_columns)`. Each row represents one order record, while each column represents a recorded attribute about that order. Dataset dimensions matter in ML projects because they establish how many observations and variables are available before later validation and modeling decisions.

In [3]:
orders.shape

(99441, 8)

### 2.2 Column Names

`DataFrame.columns` contains the labels of all available columns. Reviewing these names before feature engineering helps us recognize identifiers, possible predictors, and columns related to future target construction. Calling `.tolist()` presents the labels as a simple Python list.

In [4]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

### 2.3 Column Data Types

`DataFrame.dtypes` reports the data type Pandas assigned to each column. Data types determine which calculations and transformations are valid and later influence preprocessing choices for ML. Because this CSV was loaded without explicit date parsing, timestamp columns may initially appear as `object` (or `str` in newer Pandas versions), meaning Pandas currently treats their values as strings rather than datetime objects.

In [5]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

### 2.4 Learning Questions

1. What does one row of the orders dataset represent?
2. How many rows and columns are present?
3. Which columns are identifiers?
4. Which columns represent dates or timestamps?
5. Are the timestamp columns currently stored as datetime objects or strings?
6. Why is knowing the data types important before performing calculations?

## 3. Uniqueness and Duplicate Inspection

Uniqueness means that a key identifies only one record in a relational dataset. Because the orders table is expected to contain one row per order, `order_id` should uniquely identify each row. Repeated identifiers can represent more than one record for the same order, while fully duplicated rows are records whose values match across every column. Either issue can distort counts, summaries, and ML training by giving some observations extra weight. We inspect both forms first so their cause and meaning can be understood before any decision about removal.

### 3.1 Count Unique Order IDs

`nunique()` counts the distinct values in a Series. By default, it excludes missing values from the count.

In [6]:
orders["order_id"].nunique()

99441

### 3.2 Verify Order ID Uniqueness

`is_unique` checks whether every value in a Series occurs only once. It is a property, so it is used without parentheses. `True` means no value is repeated; `False` means at least one value is repeated.

In [7]:
orders["order_id"].is_unique

True

### 3.3 Count Duplicated Order IDs

`duplicated()` returns a Boolean Series that marks repeated occurrences. By default, the first occurrence is not marked as duplicated. Calling `.sum()` counts the `True` values, so the result measures repeated occurrences rather than the number of distinct IDs that repeat. This inspection does not delete or filter any rows. The result is displayed as `np.int64(0)` because Pandas returns a NumPy integer; it simply means the count is 0.

In [8]:
orders["order_id"].duplicated().sum()

np.int64(0)

### 3.4 Count Fully Duplicated Rows

A duplicated identifier repeats the key even when other column values differ. A fully duplicated row matches an earlier row across every column. `orders.duplicated()` checks complete rows, and `.sum()` counts the repeated occurrences without changing the DataFrame. The result is displayed as `np.int64(0)` because Pandas returns a NumPy integer; it simply means the count is 0.

In [9]:
orders.duplicated().sum()

np.int64(0)

### 3.5 Learning Questions

1. What is the difference between the total number of rows and the number of unique order IDs?
2. Why should `order_id` be unique in the orders table?
3. What is the difference between a duplicated order ID and a completely duplicated row?
4. Why might `order_id` legitimately appear multiple times in the `order_items` table?
5. How could accidental duplicate records affect ML model training?
6. Why should we investigate duplicate records before removing them?

## 4. Missing Values Investigation — Products

This section investigates missing product attributes without cleaning, replacing, or removing any values. The goal is to understand which fields are incomplete, whether missing values occur together, and why those patterns may matter later for delivery prediction.

### 4.1 Load and Preview the Products Dataset

The existing `project_root` and Pandas import are reused. The products CSV path follows the same project-relative convention as the orders path, and the existence check reports the expected location if the file is unavailable.

In [10]:
products_path = project_root / "data" / "raw" / "olist_products_dataset.csv"

if not products_path.exists():
    raise FileNotFoundError(f"Products dataset not found at expected path: {products_path}")

products = pd.read_csv(products_path)

The shape confirms how many product records and columns were loaded. The small preview checks the column layout without changing the data.

In [11]:
products.shape

(32951, 9)

In [12]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### 4.2 Identify Missing Values

`isna()` returns `True` wherever Pandas recognizes a missing value. For Boolean values, `.sum()` counts the `True` entries, producing the number of missing values in each column. Because `True` behaves like 1 and `False` like 0, `.mean() * 100` gives the percentage missing in each column. With `axis=1`, Pandas checks across the columns of each row, so the final count measures products with at least one missing value.

In [13]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [14]:
products.isna().mean() * 100

product_id                    0.000000
product_category_name         1.851234
product_name_lenght           1.851234
product_description_lenght    1.851234
product_photos_qty            1.851234
product_weight_g              0.006070
product_length_cm             0.006070
product_height_cm             0.006070
product_width_cm              0.006070
dtype: float64

In [15]:
products.isna().any(axis=1).sum()

np.int64(611)

### 4.3 Inspect Incomplete Product Records

Filtering with the row-level missing-value mask selects product records with at least one missing field. `.head()` keeps the displayed sample small and readable; it does not remove records from `products`.

In [16]:
products[products.isna().any(axis=1)].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


### 4.4 Investigate Missingness Patterns

The descriptive and physical columns are examined as two separate groups. Boolean masks show whether each product is missing any field in a group, several descriptive fields, all descriptive fields, or fields from both groups. The summary reports counts and percentages without changing the source DataFrame.

In [17]:
descriptive_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
]
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

descriptive_missing = products[descriptive_columns].isna()
physical_missing = products[physical_columns].isna()
has_descriptive_missing = descriptive_missing.any(axis=1)
has_multiple_descriptive_missing = descriptive_missing.sum(axis=1) > 1
has_all_descriptive_missing = descriptive_missing.all(axis=1)
has_physical_missing = physical_missing.any(axis=1)
has_descriptive_and_physical_missing = (
    has_descriptive_missing & has_physical_missing
)
has_complete_record = ~products.isna().any(axis=1)

In [18]:
missingness_pattern_summary = pd.DataFrame(
    {
        "product_count": [
            has_descriptive_missing.sum(),
            has_multiple_descriptive_missing.sum(),
            has_all_descriptive_missing.sum(),
            has_physical_missing.sum(),
            has_descriptive_and_physical_missing.sum(),
            has_complete_record.sum(),
        ]
    },
    index=[
        "Any descriptive attribute missing",
        "Multiple descriptive attributes missing",
        "All descriptive attributes missing",
        "Any physical measurement missing",
        "Descriptive and physical missingness overlap",
        "Complete product records",
    ],
)
missingness_pattern_summary["percentage"] = (
    missingness_pattern_summary["product_count"] / len(products) * 100
)
missingness_pattern_summary

,product_count,percentage
Any descriptive attribute missing,610,1.851234
Multiple descriptive attributes missing,610,1.851234
All descriptive attributes missing,610,1.851234
Any physical measurement missing,2,0.006070
Descriptive and physical missingness overlap,1,0.003035
Complete product records,32340,98.145732


### 4.5 NaN Versus Zero

**NaN** represents a missing or unknown recorded value. It does not automatically mean zero, and Pandas generally recognizes it as missing data. For example, a missing `product_photos_qty` does not tell us that the product has no photos.

**Zero** is a numeric value explicitly recorded as zero. Pandas does not normally classify it as missing. A zero photo count or zero product weight may be valid, invalid, or suspicious depending on the business definition, so these values are counted separately without being labeled as errors.

In [19]:
products["product_photos_qty"].isna().sum()

np.int64(610)

In [20]:
(products["product_photos_qty"] == 0).sum()

np.int64(0)

In [21]:
(products["product_weight_g"] == 0).sum()

np.int64(4)

### 4.6 ML Relevance

**Observed facts:** the preceding outputs measure missingness in product category, descriptive metadata, weight, and dimensions at the product-record level. They do not yet measure how many orders contain those products.

**Potential modeling implications to validate later:** weight and dimensions could help explain delivery duration, while category could contribute to either delivery-duration or late-delivery prediction. Missing descriptive attributes may also act as data-quality indicators. These are candidate interpretations, not established predictive effects. Because products can appear in multiple order items, a small share of incomplete products could affect a different share of orders after joining. That order-level impact must be measured with a duplicate-safe join in a later task; no feature engineering or joining is performed here.

### 4.7 Learning Questions

1. What is the difference between `NaN` and a recorded zero?
2. Why does `products.isna().sum()` count missing values?
3. Why does `products.isna().mean() * 100` give missing percentages?
4. What does `axis=1` mean in the incomplete-row check?
5. Why might several attributes be missing for the same product?
6. Why could replacing missing product weight with zero be misleading?
7. What happens if we remove products with missing attributes?
8. Why might a small percentage of missing products affect many orders?

## Complete Nine-Dataset Understanding (Before Task 5)

The earlier sections are preserved as Tasks 1–4. This section extends the same read-only investigation to all nine source files without cleaning, joining, engineering features, or constructing targets. The final modeling grain will later be one row per `order_id`, but the source tables intentionally have different grains.

The prediction boundary is **order placement** (`order_purchase_timestamp`). Fields observed after purchase may be useful for labels or retrospective data-quality checks, but they are not valid purchase-time features.

### A. Load the Remaining Source Datasets

`orders` and `products` are reused from the earlier tasks. The seven remaining files use the same project-relative path convention and are each loaded once. The existence check reports all missing inputs together before Pandas attempts to read them.

In [22]:
raw_data_dir = project_root / "data" / "raw"

dataset_files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    filename
    for filename in dataset_files.values()
    if not (raw_data_dir / filename).exists()
]
if missing_files:
    raise FileNotFoundError(f"Missing source CSV files: {missing_files}")

order_items = pd.read_csv(raw_data_dir / dataset_files["order_items"])
customers = pd.read_csv(raw_data_dir / dataset_files["customers"])
sellers = pd.read_csv(raw_data_dir / dataset_files["sellers"])
payments = pd.read_csv(raw_data_dir / dataset_files["payments"])
reviews = pd.read_csv(raw_data_dir / dataset_files["reviews"])
geolocation = pd.read_csv(raw_data_dir / dataset_files["geolocation"])
category_translation = pd.read_csv(
    raw_data_dir / dataset_files["category_translation"]
)

datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Customers": customers,
    "Sellers": sellers,
    "Payments": payments,
    "Reviews": reviews,
    "Geolocation": geolocation,
    "Category Translation": category_translation,
}

### B. Dataset Overview

The dimensions below are computed from the loaded DataFrames rather than copied from documentation. A dataset's row count should always be interpreted together with its grain.

In [23]:
dataset_purposes = {
    "Orders": "Order status and purchase, delivery, and estimate timestamps",
    "Order Items": "Products, sellers, price, and freight recorded within orders",
    "Products": "Product category and physical metadata",
    "Customers": "Order-specific customer identifiers and location",
    "Sellers": "Seller identifiers and location",
    "Payments": "Payment records and installment details associated with orders",
    "Reviews": "Post-purchase ratings, comments, and review timestamps",
    "Geolocation": "Repeated coordinates and locality records for ZIP prefixes",
    "Category Translation": "Portuguese-to-English product-category lookup",
}

dataset_overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "source_file": dataset_files[name.lower().replace(" " , "_")],
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "purpose": dataset_purposes[name],
        }
        for name, dataframe in datasets.items()
    ]
)
dataset_overview

,dataset,rows,columns,purpose
0,Orders,99441,8,"Order status and purchase, delivery, and estim..."
1,Order Items,112650,7,"Products, sellers, price, and freight recorded..."
2,Products,32951,9,Product category and physical metadata
3,Customers,99441,5,Order-specific customer identifiers and location
4,Sellers,3095,4,Seller identifiers and location
5,Payments,103886,5,Payment records and installment details associ...
6,Reviews,99224,7,"Post-purchase ratings, comments, and review ti..."
7,Geolocation,1000163,5,Repeated coordinates and locality records for ...
8,Category Translation,71,2,Portuguese-to-English product-category lookup


### C. A Compact Profiling Pattern

The same small read-only helper keeps the new dataset sections consistent. It displays shape, original column names, inferred Pandas data types, a three-row sample, per-column missing counts and percentages, and the count of fully duplicated rows. Repeated identifiers are evaluated separately because they can be valid at a one-to-many grain.

In [24]:
def profile_dataset(dataframe, include_missing=True):
    """Display a compact, read-only profile of one loaded source DataFrame."""
    print(f"Shape: {dataframe.shape}")
    print(f"Columns: {dataframe.columns.tolist()}")
    display(dataframe.dtypes.rename("dtype").to_frame())
    display(dataframe.head(3))

    if include_missing:
        missing_summary = pd.DataFrame(
            {
                "missing_count": dataframe.isna().sum(),
                "missing_percentage": dataframe.isna().mean() * 100,
            }
        )
        display(missing_summary)

    print(f"Fully duplicated rows: {dataframe.duplicated().sum():,}")

### D. Orders — Lifecycle Records

**Source file:** `olist_orders_dataset.csv`  
**DataFrame:** `orders`

**Purpose and grain:** one row represents one order. The earlier sections already establish the shape, schema, sample, `order_id` uniqueness, and absence of fully duplicated rows. The focused checks below add per-column missingness, observed statuses, timestamp identification, and a descriptive status-by-missing-timestamp view.

`order_purchase_timestamp` defines the future prediction boundary. `order_estimated_delivery_date` is the historical promise and a possible purchase-time input after its availability is validated. Approval, carrier handoff, actual delivery, and status outcomes occur after purchase or can reveal outcomes, so they must not be purchase-time features. No target is calculated here.

In [25]:
orders_missing_summary = pd.DataFrame(
    {
        "missing_count": orders.isna().sum(),
        "missing_percentage": orders.isna().mean() * 100,
    }
)
orders_missing_summary

,missing_count,missing_percentage
order_id,0,0.000000
customer_id,0,0.000000
order_status,0,0.000000
order_purchase_timestamp,0,0.000000
order_approved_at,160,0.160899
order_delivered_carrier_date,1783,1.793023
order_delivered_customer_date,2965,2.981668
order_estimated_delivery_date,0,0.000000


In [26]:
order_timestamp_columns = [
    column
    for column in orders.columns
    if "timestamp" in column or "date" in column
]

print("Observed order statuses:")
display(orders["order_status"].value_counts().rename("order_count"))
print("Timestamp-related columns:")
display(order_timestamp_columns)
print("Missing timestamp counts by recorded order status:")
orders.groupby("order_status")[order_timestamp_columns].agg(
    lambda column: column.isna().sum()
)

Observed order statuses:


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_count, dtype: int64

Timestamp-related columns:


['order_purchase_timestamp',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

Missing timestamp counts by recorded order status:


,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,
approved,0,2,2,0
canceled,0,550,619,0
created,0,5,5,0
delivered,0,2,8,0
invoiced,0,314,314,0
processing,0,301,301,0
shipped,0,0,1107,0
unavailable,0,609,609,0


The status grouping is descriptive: it shows where timestamps are absent but does not declare every absence erroneous. For example, an undelivered order may legitimately lack an actual delivery timestamp; exact validity rules belong in `02_data_quality.ipynb`.

#### Orders — Summary of What the Table Shows

> **In plain language:** This is the central table for the project. It contains 99,441 orders and eight columns, with one unique `order_id` per row and no fully duplicated rows. There are eight recorded order statuses. A total of 2,980 rows contain at least one missing value, mainly because some approval, carrier-handoff, or customer-delivery timestamps are absent. Those missing lifecycle timestamps must be interpreted together with `order_status`; they are not automatically errors. All later tables connect directly or indirectly back to these orders.

### E. Order Items — Products and Sellers Within Orders

**Source file:** `olist_order_items_dataset.csv`  
**DataFrame:** `order_items`

**Purpose and grain:** one row represents an item sequence within an order. An order can have several item rows, while `order_item_id` is a within-order sequence rather than a global identifier. The candidate composite key is (`order_id`, `order_item_id`). `product_id` and `seller_id` connect each item to its product and seller. Price, freight, product, and seller information may be relevant to both ML problems if confirmed available at purchase; `shipping_limit_date` remains availability-uncertain.

In [27]:
profile_dataset(order_items)

order_items_identifier_checks = pd.Series(
    {
        "unique order_id values": order_items["order_id"].nunique(),
        "repeated order_id occurrences": order_items["order_id"].duplicated().sum(),
        "order_id is unique": order_items["order_id"].is_unique,
        "(order_id, order_item_id) is unique": ~order_items.duplicated(
            subset=["order_id", "order_item_id"]
        ).any(),
        "unique product_id values": order_items["product_id"].nunique(),
        "unique seller_id values": order_items["seller_id"].nunique(),
    }
)
display(order_items_identifier_checks)
display(order_items[["order_item_id", "price", "freight_value"]].describe())

Shape: (112650, 7)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


,dtype
order_id,str
order_item_id,int64
product_id,str
seller_id,str
shipping_limit_date,str
price,float64
freight_value,float64


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


,missing_count,missing_percentage
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


Fully duplicated rows: 0


unique order_id values                 98666
repeated order_id occurrences          13984
order_id is unique                     False
(order_id, order_item_id) is unique     True
unique product_id values               32951
unique seller_id values                 3095
dtype: object

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


#### Key Findings — Order Items Statistical Summary

1. **Dataset completeness:** All 112,650 order-item records have non-missing values for `order_item_id`, `price`, and `freight_value`.
2. **Product price distribution:** The average item price is R$120.65, while the median is R$74.99. Since the mean is higher than the median, the price distribution appears right-skewed, suggesting that expensive items pull the average upward.
3. **Typical product prices:** Approximately 50% of item records have prices between R$39.90 (25th percentile) and R$134.90 (75th percentile).
4. **Potential price outliers:** The maximum item price is R$6,735, considerably higher than the 75th percentile. These unusually expensive records require further investigation before being classified as outliers or errors.
5. **Freight distribution:** The average freight charge is R$19.99, while the median is R$16.26, suggesting a right-skewed distribution. The maximum freight charge is R$409.68, which may warrant investigation.
6. **Zero freight values:** The minimum freight value is R$0. This represents a recorded zero charge, not a missing value.
7. **Order item sequence:** The median `order_item_id` is 1, and the maximum is 21. Some orders therefore have item positions beyond the first item. However, `order_item_id` is a sequence identifier, not the total number of items per order.

#### ML Relevance

Item price, freight cost, and the number of items per order are potential features for delivery-time regression and late-delivery classification.

Before modeling, we should investigate their distributions, validate unusual values, and aggregate item-level information to the order level.

**Next investigation:** Examine price and freight distributions using histograms and box plots, and calculate the number of items per order.

#### Order Items — Summary of What the Table Shows

> **In plain language:** This table explains what was purchased and who sold it. Its 112,650 rows belong to 98,666 orders, so repeated `order_id` values are expected rather than accidental duplicates. The combination of `order_id` and `order_item_id` uniquely identifies every item row. There are no missing values or fully duplicated rows in this snapshot. Before building an order-level dataset, item price, freight, product, and seller information must be summarized per order so multi-item orders do not create multiple modeling rows.

### F. Products — Product Metadata

**Source file:** `olist_products_dataset.csv`  
**DataFrame:** `products`

**Purpose and grain:** one row represents one `product_id`. Product category, weight, and dimensions could become order-level candidates only after items are joined and safely aggregated in a later phase. The source spellings `product_name_lenght` and `product_description_lenght` are intentionally preserved.

Task 4 already provides the detailed schema preview, per-column missingness, incomplete-row sample, missingness patterns, and NaN-versus-zero comparison. The concise checks below add only the missing identifier and duplicate evidence; the full missingness analysis is not repeated.

In [28]:
pd.Series(
    {
        "unique product_id values": products["product_id"].nunique(),
        "product_id is unique": products["product_id"].is_unique,
        "fully duplicated rows": products.duplicated().sum(),
    }
)

unique product_id values    32951
product_id is unique         True
fully duplicated rows           0
dtype: object

#### Products — Summary of What the Table Shows

> **In plain language:** This is a product reference table containing 32,951 unique products and nine columns. `product_id` is unique, and there are no fully duplicated rows. However, 611 products have at least one missing attribute: 610 are missing category or descriptive metadata, while two are missing physical measurements. These missing values are not the same as recorded zeros, as demonstrated in Task 4. Product attributes can only reach the final order-level data through order items and may need aggregation when an order contains several products.

### G. Customers — Order-Specific and Cross-Order Identity

**Source file:** `olist_customers_dataset.csv`  
**DataFrame:** `customers`

**Purpose and grain:** one row represents an order-specific `customer_id`. That key joins directly to `orders.customer_id`. `customer_unique_id` is a different, cross-order identity and can repeat across customer records; it must not replace `customer_id` in the orders join. ZIP prefix, city, and state are geographic candidates for both prediction problems, subject to inference availability and later quality validation.

In [29]:
profile_dataset(customers)

customer_record_counts = customers["customer_unique_id"].value_counts()
pd.Series(
    {
        "unique customer_id values": customers["customer_id"].nunique(),
        "customer_id is unique": customers["customer_id"].is_unique,
        "unique customer_unique_id values": customers["customer_unique_id"].nunique(),
        "customer_unique_id is unique": customers["customer_unique_id"].is_unique,
        "unique customers with multiple records": (customer_record_counts > 1).sum(),
        "maximum records for one customer_unique_id": customer_record_counts.max(),
    }
)

Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


,missing_count,missing_percentage
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


Fully duplicated rows: 0


unique customer_id values                     99441
customer_id is unique                          True
unique customer_unique_id values              96096
customer_unique_id is unique                  False
unique customers with multiple records         2997
maximum records for one customer_unique_id       17
dtype: object

#### Customers — Summary of What the Table Shows

> **In plain language:** This table has 99,441 rows with a unique `customer_id` in every row, matching the order-specific customer key used by the orders table. It contains 96,096 distinct `customer_unique_id` values; 2,997 of those identities appear in multiple customer records, showing that the same underlying customer can place more than one order. The table has no missing or fully duplicated rows. Use `customer_id` for the orders join and reserve `customer_unique_id` for carefully designed cross-order customer analysis.

### H. Sellers — Seller Location Reference

**Source file:** `olist_sellers_dataset.csv`  
**DataFrame:** `sellers`

**Purpose and grain:** one row represents one seller. `seller_id` links from order items. Seller ZIP prefix, city, and state may describe fulfillment geography for both ML problems, but multi-seller orders will require explicit order-level aggregation later.

In [30]:
profile_dataset(sellers)

pd.Series(
    {
        "unique seller_id values": sellers["seller_id"].nunique(),
        "seller_id is unique": sellers["seller_id"].is_unique,
        "unique seller ZIP prefixes": sellers["seller_zip_code_prefix"].nunique(),
        "unique seller cities": sellers["seller_city"].nunique(),
        "unique seller states": sellers["seller_state"].nunique(),
    }
)

Shape: (3095, 4)
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


,dtype
seller_id,str
seller_zip_code_prefix,int64
seller_city,str
seller_state,str


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


,missing_count,missing_percentage
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


Fully duplicated rows: 0


unique seller_id values       3095
seller_id is unique           True
unique seller ZIP prefixes    2246
unique seller cities           611
unique seller states            23
dtype: object

#### Sellers — Summary of What the Table Shows

> **In plain language:** This is a compact seller reference table with 3,095 rows and four columns. Every `seller_id` is unique, and no rows contain missing values or complete duplicates. The remaining columns describe seller ZIP prefix, city, and state. Sellers connect to orders through the order-items table, and one order can involve multiple sellers, so seller information will eventually require an explicit order-level summary.

### I. Payments — Payment Sequences Within Orders

**Source file:** `olist_order_payments_dataset.csv`  
**DataFrame:** `payments`

**Purpose and grain:** one row represents a payment sequence associated with an order. `order_id` can repeat because an order may use several payment records; (`order_id`, `payment_sequential`) is the candidate composite key. Payment type, installments, and value might be relevant only if they are finalized and available at order placement, which remains an inference-contract question.

In [31]:
profile_dataset(payments)

payment_identifier_checks = pd.Series(
    {
        "unique order_id values": payments["order_id"].nunique(),
        "repeated order_id occurrences": payments["order_id"].duplicated().sum(),
        "order_id is unique": payments["order_id"].is_unique,
        "(order_id, payment_sequential) is unique": ~payments.duplicated(
            subset=["order_id", "payment_sequential"]
        ).any(),
    }
)
display(payment_identifier_checks)
display(payments["payment_type"].value_counts().rename("payment_record_count"))
display(payments[["payment_sequential", "payment_installments", "payment_value"]].describe())

Shape: (103886, 5)
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


,dtype
order_id,str
payment_sequential,int64
payment_type,str
payment_installments,int64
payment_value,float64


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


,missing_count,missing_percentage
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


Fully duplicated rows: 0


unique order_id values                      99440
repeated order_id occurrences                4446
order_id is unique                          False
(order_id, payment_sequential) is unique     True
dtype: object

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: payment_record_count, dtype: int64

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


#### Key Findings — Payments Statistical Summary

1. **Numeric completeness:** All 103,886 payment records have non-missing values for `payment_sequential`, `payment_installments`, and `payment_value`.
2. **Payment sequences:** The median `payment_sequential` is 1, while the maximum is 29. This indicates that some orders contain multiple payment records.
3. **Installment distribution:** The average installment count is 2.85, while the median is 1. The maximum is 24 installments. Higher installment counts increase the average.
4. **Zero installments:** The minimum installment count is 0. These records require further investigation before being treated as invalid.
5. **Payment-value distribution:** The average payment value is R$154.10, while the median is R$100.00. This suggests a right-skewed distribution.
6. **Typical payment range:** Approximately 50% of payment records have values between R$56.79 (25th percentile) and R$171.84 (75th percentile).
7. **Potential extreme values:** The maximum payment value is R$13,664.08, substantially higher than the 75th percentile. These observations require investigation before being classified as errors or outliers.
8. **Zero payment values:** The minimum payment value is R$0. Recorded zero values are not the same as missing values.

#### ML Relevance

Payment amount, payment method, installment count, and number of payment records per order may be potential features for delivery-time regression and late-delivery classification, provided they are available at the defined prediction time.

Because an order may contain multiple payment records, payment information must be aggregated appropriately before constructing the final order-level modeling dataset.

**Further investigation:** Examine payment types, zero installments, zero payment amounts, multiple payment records per order, and extreme payment values during data-quality analysis.

#### Payments — Summary of What the Table Shows

> **In plain language:** The 103,886 payment rows correspond to 99,440 orders, confirming that some orders have more than one payment record. Repeated `order_id` values are therefore expected. The combination of `order_id` and `payment_sequential` is unique, and the table has no missing or fully duplicated rows. It records payment type, installment count, and value, but these fields cannot become model inputs until their availability at the exact purchase-time prediction boundary is confirmed.

### J. Reviews — Post-Purchase Feedback

**Source file:** `olist_order_reviews_dataset.csv`  
**DataFrame:** `reviews`

**Purpose and grain:** one row represents a review record, but neither `review_id` nor `order_id` is individually unique in this extract. The candidate pair (`review_id`, `order_id`) is checked empirically, while its business meaning still requires validation. Review scores, comments, and timestamps describe post-purchase experience and are prohibited as purchase-time model inputs, though they remain useful for retrospective understanding.

In [32]:
profile_dataset(reviews)

review_identifier_checks = pd.Series(
    {
        "unique review_id values": reviews["review_id"].nunique(),
        "review_id is unique": reviews["review_id"].is_unique,
        "unique order_id values": reviews["order_id"].nunique(),
        "order_id is unique": reviews["order_id"].is_unique,
        "(review_id, order_id) is unique": ~reviews.duplicated(
            subset=["review_id", "order_id"]
        ).any(),
    }
)
display(review_identifier_checks)
display(reviews["review_score"].value_counts().sort_index().rename("review_count"))
display(reviews[["review_creation_date", "review_answer_timestamp"]].head(3))

Shape: (99224, 7)
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


,dtype
review_id,str
order_id,str
review_score,int64
review_comment_title,str
review_comment_message,str
review_creation_date,str
review_answer_timestamp,str


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


,missing_count,missing_percentage
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


Fully duplicated rows: 0


unique review_id values            98410
review_id is unique                False
unique order_id values             98673
order_id is unique                 False
(review_id, order_id) is unique     True
dtype: object

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: review_count, dtype: int64

,review_creation_date,review_answer_timestamp
0,2018-01-18 00:00:00,2018-01-18 21:46:59
1,2018-03-10 00:00:00,2018-03-11 03:05:13
2,2018-02-17 00:00:00,2018-02-18 14:36:24


The missing fields are the optional comment title and message columns. Their high missing percentages describe sparsity, not automatically a data defect. Reviews also do not form a perfect one-row-per-order table, so a raw join could increase order rows.

#### Reviews — Summary of What the Table Shows

> **In plain language:** This table contains 99,224 review records covering 98,673 orders. Neither `review_id` nor `order_id` is individually unique, although their combination is unique in this snapshot. There are no fully duplicated rows, but 89,385 rows have at least one missing value because review titles and messages are optional and highly sparse. Review scores, text, and timestamps describe the experience after purchase, so they are useful for retrospective analysis but would leak future information into a purchase-time model.

### K. Geolocation — Repeated ZIP-Prefix Observations

**Source file:** `olist_geolocation_dataset.csv`  
**DataFrame:** `geolocation`

**Purpose and grain:** one row is a raw coordinate/locality observation associated with a ZIP-code prefix, not a unique ZIP lookup row. A ZIP prefix can have multiple coordinate records, and complete rows can also repeat. Latitude and longitude may later support geographic distance or regional signals, but raw geolocation must first be reduced to a documented unique lookup grain in a later phase.

In [33]:
profile_dataset(geolocation)

geolocation_records_per_zip = geolocation[
    "geolocation_zip_code_prefix"
].value_counts()
pd.Series(
    {
        "unique ZIP prefixes": geolocation[
            "geolocation_zip_code_prefix"
        ].nunique(),
        "ZIP prefix is unique": geolocation[
            "geolocation_zip_code_prefix"
        ].is_unique,
        "repeated ZIP-prefix occurrences": geolocation[
            "geolocation_zip_code_prefix"
        ].duplicated().sum(),
        "ZIP prefixes with multiple records": (
            geolocation_records_per_zip > 1
        ).sum(),
        "maximum records for one ZIP prefix": geolocation_records_per_zip.max(),
        "fully duplicated rows": geolocation.duplicated().sum(),
    }
)

Shape: (1000163, 5)
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


,missing_count,missing_percentage
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


Fully duplicated rows: 261,831


unique ZIP prefixes                    19015
ZIP prefix is unique                   False
repeated ZIP-prefix occurrences       981148
ZIP prefixes with multiple records     17972
maximum records for one ZIP prefix      1146
fully duplicated rows                 261831
dtype: object

#### Geolocation — Summary of What the Table Shows

> **In plain language:** This is the largest source table, with 1,000,163 coordinate/locality observations but only 19,015 distinct ZIP prefixes. ZIP prefix is therefore not a primary key; one prefix can have as many as 1,146 records. The table contains no missing values, but it has 261,831 fully duplicated rows. Joining customers or sellers directly to this raw table would multiply records, so a documented one-row-per-ZIP lookup must be designed and validated later.

### L. Category Translation — Portuguese to English

**Source file:** `product_category_name_translation.csv`  
**DataFrame:** `category_translation`

**Purpose and grain:** one row maps a Portuguese `product_category_name` to an English label. Both columns are tested for uniqueness and missingness. The Portuguese key relates to `products.product_category_name`; products can repeat a category, and products with a missing or unmatched category would not gain a translation from an inner lookup.

In [34]:
profile_dataset(category_translation)

untranslated_product_categories = sorted(
    set(products["product_category_name"].dropna())
    - set(category_translation["product_category_name"])
)

pd.Series(
    {
        "Portuguese category key is unique": category_translation[
            "product_category_name"
        ].is_unique,
        "English category values are unique": category_translation[
            "product_category_name_english"
        ].is_unique,
        "missing Portuguese keys": category_translation[
            "product_category_name"
        ].isna().sum(),
        "missing English translations": category_translation[
            "product_category_name_english"
        ].isna().sum(),
        "non-null product categories without a translation": len(
            untranslated_product_categories
        ),
    }
)

Shape: (71, 2)
Columns: ['product_category_name', 'product_category_name_english']


,dtype
product_category_name,str
product_category_name_english,str


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


,missing_count,missing_percentage
product_category_name,0,0.0
product_category_name_english,0,0.0


Fully duplicated rows: 0


Portuguese category key is unique                    True
English category values are unique                   True
missing Portuguese keys                                 0
missing English translations                            0
non-null product categories without a translation       2
dtype: object

In [35]:
untranslated_product_categories

['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

#### Category Translation — Summary of What the Table Shows

> **In plain language:** This small lookup contains 71 Portuguese category keys and their English translations. Both columns are unique, with no missing values or fully duplicated rows. It can add readable English labels to the products table through `product_category_name`, but two non-null product categories in the products data do not have a matching translation. A later join should preserve those original Portuguese values instead of silently dropping the affected products.

## Dataset Relationships

The table below documents intended join paths without building the production join. Uniqueness statements are supported by the checks above. “Can increase rows?” is evaluated from the perspective of retaining the left table's current grain.

| Relationship | Join column(s) | Observed key behavior | Expected relationship | Can increase left rows? | Caveat for later validation |
|---|---|---|---|---|---|
| Orders → Customers | `orders.customer_id = customers.customer_id` | Unique in both extracted tables | One matched customer record per order in this snapshot | Not if customer uniqueness continues to hold | Use `customer_id`, not `customer_unique_id`; validate unmatched keys |
| Orders → Order Items | `order_id` | Unique in orders; repeated in items | One-to-many | Yes | Aggregate items to one row per order before the modeling join |
| Order Items → Products | `product_id` | Repeated in items; unique in products | Many-to-one | No if product uniqueness holds | Validate unmatched product keys before enrichment |
| Order Items → Sellers | `seller_id` | Repeated in items; unique in sellers | Many-to-one | No if seller uniqueness holds | One order may still contain multiple sellers after item enrichment |
| Orders → Payments | `order_id` | Unique in orders; repeated in payments | One-to-many | Yes | Aggregate payments independently; purchase-time availability is unresolved |
| Orders → Reviews | `order_id` | Unique in orders; repeated in reviews | Zero/one-to-many | Yes | Reviews are post-purchase and excluded from purchase-time features |
| Customers/Sellers → Geolocation | ZIP-prefix columns | ZIP values repeat on both business rows and raw geolocation | Many-to-many if joined raw | Yes, potentially substantially | Build and validate one canonical geolocation row per ZIP first |
| Products → Category Translation | `product_category_name` | Repeated in products; unique in translation | Many-to-one for mapped non-null categories | No if translation key remains unique | Preserve missing/untranslated Portuguese categories in later left joins |

Raw one-to-many tables must not be joined to one another before independent order-level aggregation: combining items and payments directly on `order_id`, for example, can multiply rows and corrupt counts or sums.

## Consolidated Missing-Values Summary

Missing cells and rows containing missing values are different metrics. A row with three missing cells must count once—not three times—in `rows_with_missing`. The summary therefore uses `isna().any(axis=1)` for row-level counts. Missingness can reflect optional or inapplicable fields and is not automatically an error. The detailed products pattern analysis remains in Task 4 above.

In [36]:
missing_values_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "columns_with_missing": dataframe.isna().any().sum(),
            "rows_with_missing": dataframe.isna().any(axis=1).sum(),
            "rows_with_missing_percentage": (
                dataframe.isna().any(axis=1).mean() * 100
            ),
        }
        for name, dataframe in datasets.items()
    ]
)
missing_values_summary

,dataset,rows,columns,columns_with_missing,rows_with_missing,rows_with_missing_percentage
0,Orders,99441,8,3,2980,2.996752
1,Order Items,112650,7,0,0,0.000000
2,Products,32951,9,8,611,1.854268
3,Customers,99441,5,0,0,0.000000
4,Sellers,3095,4,0,0,0.000000
5,Payments,103886,5,0,0,0.000000
6,Reviews,99224,7,2,89385,90.084052
7,Geolocation,1000163,5,0,0,0.000000
8,Category Translation,71,2,0,0,0.000000


## Relevance to Regression and Classification

### Regression: actual delivery duration

Potential purchase-time information sources include the purchase timestamp, customer and seller geography, item composition, price and freight, and product weight and dimensions. Their usefulness has not been established here. Multi-item and multi-seller attributes will eventually need leakage-safe aggregation to one row per order.

### Classification: late delivery

The estimated delivery date is the historical promise used later in label construction and may also support a purchase-time promise-length feature. The actual customer-delivery timestamp is needed only to create the historical label; it must never be an input feature. Purchase-time order, item, product, seller, customer, and duplicate-safe geographic attributes are candidates subject to the future inference contract.

### Prediction-time and leakage rules

- Prediction occurs at `order_purchase_timestamp`.
- Actual delivery, carrier handoff, approval outcomes, order outcomes/status, and reviews are post-purchase information and cannot be input features.
- Target-generation fields may be retained for label construction and evaluation but must be separated from model inputs.
- Payment fields and `shipping_limit_date` remain uncertain until their purchase-time availability is established.
- Static lookups still require historical and serving-parity validation.
- No target, feature, cleaned table, production join, or model is created in this notebook.

## Key Findings

1. The nine source files cover orders, item composition, products, customers, sellers, payments, reviews, raw geolocation, and category translation; their grains differ materially.
2. `order_id` is unique in orders, but legitimately repeats in items, payments, and reviews. The item and payment composite keys tested here are unique.
3. `customer_id` is the unique order-to-customer join key in this snapshot; `customer_unique_id` represents cross-order identity and repeats.
4. Products contain 611 rows with at least one missing field. Orders contain 2,980 incomplete rows, driven by lifecycle timestamps. Reviews contain 89,385 rows with at least one missing optional comment field.
5. Raw geolocation is not a ZIP-prefix dimension: 1,000,163 rows contain 19,015 ZIP prefixes and 261,831 fully duplicated rows. A direct raw ZIP join risks severe row multiplication.
6. Reviews are not one-to-one with orders and neither `review_id` nor `order_id` is individually unique. Reviews are also post-purchase and excluded from purchase-time features.
7. Translation keys are unique and non-null, but two non-null product categories have no English mapping in this snapshot.
8. Items, payments, and reviews must be handled independently at their native grains before any later order-level combination.
9. Candidate ML inputs still require proof that they existed at order placement and can be reproduced at inference time.
10. `02_data_quality.ipynb` should investigate referential integrity, unmatched keys, timestamp validity and ordering, value ranges, canonical geolocation rules, category-translation coverage, and missing-child orders without silently cleaning them.

## Learning Questions

These questions are intentionally unanswered.

1. How does a dataset's grain determine which columns can form a primary key?
2. When is an identifier a primary key, a foreign key, or only part of a composite key?
3. Why are (`order_id`, `order_item_id`) and (`order_id`, `payment_sequential`) more suitable candidate keys than `order_id` alone in their tables?
4. How can a dataset have many repeated identifiers but no fully duplicated rows?
5. Why can missing-cell counts differ substantially from counts of rows containing missing values?
6. How should a one-to-many source be reduced before joining it to a one-row-per-order modeling table?
7. How can joining two one-to-many tables on `order_id` multiply rows and distort sums?
8. Why are `customer_id` and `customer_unique_id` not interchangeable?
9. When does a recorded zero carry different information from `NaN`?
10. Which fields are truly available at order placement, and how would using later lifecycle or review fields create target leakage?
11. What evidence is needed before describing a relationship as one-to-one rather than one-to-many?
12. How should repeated geolocation observations be converted into a defensible ZIP-prefix lookup without hiding uncertainty?

## Handoff to 02_data_quality.ipynb

### A. Purpose and Scope

Notebook 01 completed the initial structural and descriptive understanding of all nine original Olist CSV files. It established their schemas, source grains, candidate identifiers, missing-value baselines, duplicate behavior, basic numeric distributions, relationships, and possible relevance to purchase-time delivery prediction.

The next notebook, `notebooks/02_data_quality.ipynb` (not yet created), should investigate whether the observed data is sufficiently valid, consistent, and join-safe before data preparation or feature engineering begins.

| Notebook 01 — Data Understanding | Notebook 02 — Data Quality |
|---|---|
| Identify schemas and dataset grain | Investigate suspicious, missing, or inconsistent values |
| Understand identifiers and relationships | Validate timestamps and referential integrity |
| Count missing values and duplicates | Assess duplicate records and repeated identifiers |
| Inspect basic statistical distributions | Quantify numeric validity and potential outliers |
| Identify possible quality concerns | Evaluate join risks and recommend evidence-based treatments |

Notebook 02 is an investigation and validation phase. It should not silently clean data, engineer final features, or construct modeling targets.

### B. Confirmed Dataset Baseline

The following values were independently reconciled against the current raw CSV files. Composite-key uniqueness is an observed property of this snapshot, not an assumed permanent business constraint.

| Dataset | Rows | Rows with missing values | Verified key observation |
|---|---:|---:|---|
| Orders | 99,441 | 2,980 | `order_id` is unique |
| Order Items | 112,650 | 0 | (`order_id`, `order_item_id`) is empirically unique |
| Products | 32,951 | 611 | `product_id` is unique |
| Customers | 99,441 | 0 | `customer_id` is unique; `customer_unique_id` is not |
| Sellers | 3,095 | 0 | `seller_id` is unique |
| Payments | 103,886 | 0 | (`order_id`, `payment_sequential`) is empirically unique |
| Reviews | 99,224 | 89,385 | Neither `review_id` nor `order_id` is unique; (`review_id`, `order_id`) is empirically unique |
| Geolocation | 1,000,163 | 0 | ZIP prefix is not unique; 261,831 complete duplicate rows exist |
| Category Translation | 71 | 0 | Portuguese `product_category_name` is unique |

Additional confirmed evidence: eight orders recorded as `delivered` lack `order_delivered_customer_date`; two non-null product categories—`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`—lack English translation keys. These are observations to investigate, not automatic deletion rules.

### C. Prioritized Data-Quality Investigation Backlog

#### Priority 1 — Orders and ML Target Reliability

| Dataset | Observed evidence | Unresolved quality question | Why it matters | Recommended Notebook 02 investigation |
|---|---|---|---|---|
| Orders | 2,980 rows contain missing values; eight delivered-status orders lack customer-delivery timestamps | How do missing purchase, approval, carrier, delivery, and estimate fields vary by status, and which historical rows could support each target? | Actual delivery and estimate timestamps govern regression/classification eligibility; incorrect chronology can corrupt labels | Profile missing timestamps by status; validate parsing and purchase → approval → carrier → customer chronology; inspect estimated-date validity; separately describe canceled, unavailable, shipped, and undelivered records without excluding them |

Notebook 02 must document target-eligibility questions but must not construct targets or assume every undelivered/canceled order is invalid.

#### Priority 2 — Referential Integrity and Join Safety

| Datasets | Observed evidence | Unresolved quality question | Why it matters | Recommended Notebook 02 investigation |
|---|---|---|---|---|
| Orders → Customers; Orders → Items; Items → Products/Sellers; Orders → Payments/Reviews | Direct parent-key checks found no child-side orphans in these six relationships, but some orders may lack child records and one-to-many keys repeat | What is the exact coverage and cardinality in both directions, including parents without children? | Inner joins may drop orders; combining raw one-to-many sources can multiply rows and monetary totals | Quantify matched/unmatched keys, parent-without-child counts, cardinalities, and expected row counts without building the production join |
| Customers/Sellers → Geolocation | Raw ZIP prefixes repeat and geolocation is not a unique dimension | Which ZIP prefixes match, conflict, or have multiple geographic candidates? | A raw ZIP join can substantially multiply customer/seller rows | Measure coverage, unmatched prefixes, records per prefix, locality conflicts, and prospective aggregation requirements |
| Products → Category Translation | Translation keys are unique, but two non-null product categories are unmatched and 610 product category values are missing | How complete is translation coverage for products and order items? | An inner join could silently remove products or items | Quantify missing and unmatched categories and evaluate safe future left-join/fallback behavior |

Do not implement the production modeling join. Items, payments, and reviews must be validated independently before any order-level aggregation.

#### Priority 3 — Products, Prices, Freight, and Payments

| Datasets | Observed evidence | Unresolved quality question | Why it matters | Recommended Notebook 02 investigation |
|---|---|---|---|---|
| Products | 611 incomplete rows; 610 missing descriptive/category fields; two missing physical measurements; recorded zero weights exist | Are missing, zero, negative, or implausible weights/dimensions legitimate, inconsistent, or erroneous? | Physical attributes may support delivery prediction, but invalid measurements could distort aggregates | Profile missingness overlap, zero/negative values, ranges, unit plausibility, and inconsistent physical attributes without imputing |
| Order Items | No missing item/price/freight values; maximum price R$6,735; maximum freight R$409.68; zero freight exists | Which prices/freight values are statistically unusual, impossible, or inconsistent with each other? | Extreme or invalid values can dominate order-level totals | Examine distributions and robust summaries; count zero/negative values; inspect price–freight relationships and extreme records without arbitrary deletion |
| Payments | No missing numeric fields; zero installments and zero values exist; maximum payment R$13,664.08; orders may have multiple records | Are zero/extreme amounts, installment counts, payment sequences, and multi-payment patterns valid? | Payment aggregation and feature availability can affect order grain and leakage risk | Profile payment types, zeros, extremes, sequences, and per-order record counts; reconcile totals only as an investigation |

Maintain four distinct labels: **missing**, **recorded zero**, **statistically unusual**, and **confirmed error**. An extreme value is not automatically invalid.

#### Priority 4 — Geolocation Quality

| Dataset | Observed evidence | Unresolved quality question | Why it matters | Recommended Notebook 02 investigation |
|---|---|---|---|---|
| Geolocation | 1,000,163 rows, 19,015 ZIP prefixes, and 261,831 fully duplicated rows; a ZIP can map to many records | Are coordinates valid and geographically consistent within each ZIP, and what coverage exists for customers/sellers? | Raw joins can multiply rows and inconsistent coordinates can corrupt geographic features | Validate latitude/longitude ranges; inspect coordinate, city, and state disagreement within ZIPs; quantify duplicates and customer/seller coverage; compare defensible future ZIP-level representations without creating one |

Do not choose the first coordinate or implement a canonical geolocation table in Notebook 01 or as part of this handoff.

#### Priority 5 — Reviews and Category Translation

| Datasets | Observed evidence | Unresolved quality question | Why it matters | Recommended Notebook 02 investigation |
|---|---|---|---|---|
| Reviews | 89,385 rows have missing optional text; IDs repeat individually; review scores range across five observed values | Are empty comments expected, are scores valid, are review timestamps chronological, and what is order coverage? | Reviews are post-purchase and cannot be purchase-time features; repeated keys can also multiply joins | Quantify comment missingness, score validity, repeated identifiers, timestamp ordering, multiple reviews per order, and orders without reviews |
| Products and Category Translation | Two non-null product categories lack translation; product source categories can also be missing | What is the complete mapping coverage and potential row-loss risk? | Translation joins can remove or relabel products if missing/unmatched keys are mishandled | Reconcile distinct source categories to translation keys and quantify affected products/items without modifying translation data |

### D. ML Engineering Constraints to Preserve

- **Objectives:** predict actual delivery duration (regression) and lateness relative to the estimated delivery date (classification).
- **Prediction point:** `order_purchase_timestamp`.
- **Modeling grain:** exactly one row per `order_id`.
- **Potential purchase-time attributes:** purchase-time order context and candidate customer, seller, item, product, freight, and static geographic fields—only after availability and serving parity are verified.
- **Post-purchase attributes:** approval outcomes, carrier handoff, actual delivery, order outcomes/status, and reviews must not become purchase-time inputs.
- **Target sources:** actual delivery timestamps are required for historical labels but are prohibited input features.
- **Undetermined availability:** payment fields and `shipping_limit_date` must not be assumed known or final at purchase time.
- **Join safety:** items and payments contain multiple rows per order; reviews repeat order IDs; raw geolocation ZIP prefixes are non-unique. These sources must not multiply the final order-level grain.

The final feature-availability contract and target-eligibility policy still require deliberate validation and approval.

### E. Expected Outputs of Notebook 02

1. Missing-value investigation and interpretation.
2. Timestamp parsing, chronology, and consistency assessment.
3. Referential-integrity and relationship-cardinality report.
4. Duplicate-row and repeated-identifier assessment.
5. Numeric validity and potential-outlier analysis.
6. Geolocation consistency and join-coverage assessment.
7. Review and category-translation coverage assessment.
8. Prioritized data-quality issue register, separating confirmed errors, possible risks, and legitimate missing/unusual values.
9. Evidence-supported data-preparation recommendations.
10. Explicit unresolved issues and their implications for targets, features, joins, and order-level coverage.

### F. Scope Boundaries

Notebook 02 should not automatically modify raw CSV files, delete observations, impute missing values, remove duplicates, apply arbitrary outlier thresholds, create a production-ready joined dataset, engineer final ML features, construct final ML labels, or train regression/classification models. Recommendations must be supported by measured evidence before later preparation decisions are adopted.

### G. Completion and Acceptance Criteria

Notebook 02 is ready for review when important quality risks are reproducibly measured; missingness is interpreted; suspicious values and timestamps are investigated; referential-integrity and join risks are quantified; impacts on ML targets and candidate features are explained; preparation recommendations are justified; and unresolved questions are recorded explicitly. Completion does not require every suspicious record to be removed or corrected.